# Verification notebook — reproduce the paper's numbers from the repository

Paper: *Making a Legacy PHP Monolith Verifiable for a Gemma 4 Bug-Fixing Agent* (`docs/KAGGLE_FINAL_WRITEUP.md`).

Every cell below recomputes a number reported in the paper **from the files committed in the repository** (run traces,
verdicts, training logs, training-data snapshots). No GPU, no Docker, no API key and no model call are needed:
CPU only, about one minute. Each cell ends with `assert` statements, so a silent mismatch is impossible.

Re-running the agent itself (Docker PrestaShop + Gemma) is optional and described in the last section.

In [ ]:
import os, subprocess, json, csv, re, random, tarfile, collections
REPO = "https://github.com/ba-rem26007/gemma4-legacy-replay.git"
if not os.path.exists("bench/catalog.jsonl"):
    if not os.path.exists("gemma4-legacy-replay"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("gemma4-legacy-replay")
print("repository:", os.getcwd())

## 1. Main results table (33 TEST bugs, Gemma 4 31B / 26B A4B) — conditions A, R, C, B, O, A-26B, E
`bench/results.py` rebuilds `docs/RESULTATS.md` and `eval/results.csv` from the verdicts stored in `runs/` (re-evaluated
verdicts preferred). We check that the regenerated files are byte-identical to the committed ones.

In [ ]:
before = {f: open(f, "rb").read() for f in ("docs/RESULTATS.md", "eval/results.csv")}
subprocess.run(["python3", "bench/results.py"], check=True, capture_output=True)
for f, b in before.items():
    assert open(f, "rb").read() == b, f"{f} differs after regeneration"
rows = list(csv.DictReader(open("eval/results.csv")))
tab = collections.defaultdict(lambda: [0, 0, set()])
for r in rows:
    t = tab[r["condition"]]; t[0] += int(r["resolu"]); t[1] += int(r["regression"]); t[2].add(r["essai"])
print(f"{'condition':10} {'solved (sum over runs)':>22} {'runs':>5} {'mean / 33':>10} {'regressions':>12}")
for c, (s, g, runs) in sorted(tab.items()):
    print(f"{c:10} {s:>22} {len(runs):>5} {s/len(runs):>10.1f} {g:>12}")
assert tab["A"][0] / 4 == 12.75 and tab["B"][0] == 15 and tab["O"][0] == 16 and tab["O"][1] == 2
print("OK: regenerated tables identical to the committed ones")

## 2. Paired bootstrap: B − A and O − A (same method as `bench/results.py`, seed 0)

In [ ]:
per = collections.defaultdict(lambda: collections.defaultdict(list))
for r in rows:
    per[r["condition"]][int(r["pr"])].append(int(r["resolu"]))
# same bug order as bench/results.py (data/bugs_test.csv, status "valide"): with a fixed seed, the bootstrap
# bounds depend on the order of the list (a different order moves them by up to ~1 point)
bugs = [int(r["pr"]) for r in csv.DictReader(open("data/bugs_test.csv")) if r["statut"] == "valide"]
assert len(bugs) == 33 and set(bugs) == set(per["A"])
import statistics
def ci(diffs):
    # exactly the bootstrap of bench/results.py: seed 0, 5000 resamples, 2.5th / 97.5th percentiles
    random.seed(0)
    boots = sorted(statistics.mean(random.choices(diffs, k=len(diffs))) for _ in range(5000))
    return statistics.mean(diffs) * 100, boots[125] * 100, boots[4875] * 100
rate = lambda c, b: sum(per[c][b]) / len(per[c][b])
def boot(c):
    return ci([rate(c, b) - rate("A", b) for b in bugs])
CI = {}
for c in ("R", "B", "O", "A-4B"):
    m, lo, hi = boot(c); CI[c] = (round(lo, 1), round(hi, 1))
    print(f"{c} − A : {m:+.1f} pts, 95% CI [{lo:+.1f}, {hi:+.1f}]")
assert CI["R"] == (-9.1, 9.1) and CI["B"] == (-2.3, 16.7) and CI["O"] == (0.8, 20.5), CI
print("OK: same intervals as in the paper")

## 3. Small model: Gemma 4 E4B base vs QLoRA adapters v15 / v16 (condition E, 3 runs each)
Raw verdicts: `runs/<run>/summary.json` (one line per bug). Details: `docs/RESULTATS_E4B.md`.

In [ ]:
E4B = {"base": ["20261001-153951-E", "20261001-184740-E", "20261001-210006-E"],
       "v15":  ["20261001-153953-E", "20261001-184742-E", "20261001-220448-E"],
       "v16":  ["20261001-153955-E", "20261001-184744-E", "20261001-215229-E"]}
R = {m: [{x["pr"]: x for x in json.load(open(f"runs/{d}/summary.json"))["results"]} for d in ds] for m, ds in E4B.items()}
assert all(len(r) == 33 for rs in R.values() for r in rs)
cnt = lambda rs, k: [sum(bool(x.get(k)) for x in r.values()) for r in rs]
for m, rs in R.items():
    print(f"{m:5} solved/run {cnt(rs,'fixed')}  applied {cnt(rs,'applied')}  right file {cnt(rs,'loc_hit')}  regressions {cnt(rs,'regression')}")
eb = bugs  # same order as section 2
erate = lambda m, b: sum(bool(r[b].get("fixed")) for r in R[m]) / 3
for m in ("v15", "v16"):
    mean, lo, hi = ci([erate(m, b) - erate("base", b) for b in eb])
    print(f"{m} − base : {mean:+.1f} pts, 95% CI [{lo:+.1f}, {hi:+.1f}]  (same bootstrap as section 2)")
    E4B_CI = globals().setdefault("E4B_CI", {}); E4B_CI[m] = (round(lo, 1), round(hi, 1))
assert E4B_CI["v15"] == (-16.2, 0.0) and E4B_CI["v16"] == (-23.2, -2.0), E4B_CI
assert cnt(R["base"], "fixed") == [3, 4, 4] and cnt(R["v15"], "fixed") == [1, 1, 2] and cnt(R["v16"], "fixed") == [0, 0, 0]
print("OK")

## 4. Training data: provenance and leakage
TEST = the 33 bugs of `data/bugs_test.csv`. A training example leaks if it is a TEST bug, was merged after the
2025-06-01 split, or edits a function also fixed by any post-split bug (`bench/catalog.jsonl`).
We check the exact snapshots used for the v15 and v16 adapters.

In [ ]:
test = {int(r["pr"]) for r in csv.DictReader(open("data/bugs_test.csv"))}
cat = {c["pr"]: c for c in map(json.loads, open("bench/catalog.jsonl"))}
post = [c for c in cat.values() if (c["merged_at"] or "9999") >= "2025-06-01"]
post_fns = {(f, fn) for c in post for f, fns in c["functions"].items() for fn in fns if not fn.startswith("(")}
def leaks(o):
    c = cat.get(o.get("pr"))
    return o.get("pr") in test or (c is not None and ((c["merged_at"] or "9999") >= "2025-06-01" or
           bool({(f, fn) for f, fns in c["functions"].items() for fn in fns} & post_fns)))
for snap in ("training/snapshots/trajectories_v15.tar.gz", "training/kaggle_dataset/trajectories.tar.gz"):
    tf = tarfile.open(snap)
    ex = [json.loads(l) for m in tf.getmembers() for l in tf.extractfile(m)]
    src = collections.Counter(o.get("source", "reconstructed") for o in ex)
    bad = [o.get("pr") for o in ex if leaks(o)]
    print(f"{snap}: {len(ex)} examples {dict(src)} — leaks: {len(bad)}")
    assert not bad
print(f"{len(test)} post-split candidates checked (the 33 evaluated TEST bugs are a subset)")
assert set(bugs) <= test
print("OK: no TEST bug and no post-split function in either training snapshot")

## 5. Fine-tuning runs (logs committed in the repository)

In [ ]:
def grab(path, pat):
    m = re.findall(pat, open(path).read()); return m[-1] if m else None
v15 = "training/lora_final/gemma-4-qlora-training-prestashop.log"
v16 = "training/lora_v16/train_v16_colab_a100.log"
print("v15 (Kaggle T4): kept", grab(v15, r"retenues pour l'entraînement : (\d+)"), "->", grab(v15, r"\(≤ 2048 tokens\) : (\d+)"),
      "examples ≤ 2048 tokens, train_loss", grab(v15, r"'train_loss': '([\d.]+)'"))
print("v16 (Colab A100): ", grab(v16, r"\(≤ 4096 tokens\) : (\d+)"), "examples ≤ 4096 tokens, train_loss", grab(v16, r"'train_loss': '([\d.]+)'"),
      "runtime", grab(v16, r"'train_runtime': '([\d.]+)'"), "s")
assert grab(v15, r"\(≤ 2048 tokens\) : (\d+)") == "89" and grab(v16, r"\(≤ 4096 tokens\) : (\d+)") == "453"
print("OK")

## 6. Self-learning loop (Gemma 4 31B only, TRAIN bugs)
Oracles written by Gemma (`bench/gentest.jsonl`, last attempt per bug) and agent runs with that oracle as feedback
(`runs/*-O`, TRAIN bugs only). The guard statistics (edits outside the official fix) need the PrestaShop history:
run `./setup.sh` then `python3 bench/loop_stats.py` (rewrites `docs/BOUCLE.md`).

In [ ]:
last = {}
for l in open("bench/gentest.jsonl"):
    if l.strip().startswith("{"):
        r = json.loads(l); last[r["pr"]] = r["statut"]
print("Gemma-written oracles:", dict(collections.Counter(last.values())))
solved = {}
for d in sorted(p for p in os.listdir("runs") if p.endswith("-O")):
    for b in os.listdir(f"runs/{d}"):
        f = f"runs/{d}/{b}/result.json"
        if b.isdigit() and int(b) not in test and os.path.exists(f):
            r = json.load(open(f)); solved[int(b)] = bool(r.get("fixed") and not r.get("regression"))
print(f"agent with Gemma oracle as feedback: {sum(solved.values())} solved / {len(solved)} TRAIN bugs")
assert collections.Counter(last.values())["valide"] == 99
print("OK")

## 7. Cost of all Gemma API calls

In [ ]:
b = json.load(open("runs/_budget.json")); print(b); assert b["eur"] == 0.0

## 8. Optional: re-run the agent end to end (Docker + Gemma API, not needed for the checks above)
```bash
./setup.sh                                                   # PrestaShop history + Playwright
cp .env.api.example .env                                     # set GEMMA_API_KEY (Google AI Studio)
bench/checkout.sh 41007 pre  && bench/replay/run.sh 41007    # hidden oracle fails on the buggy code
bench/checkout.sh 41007 post && bench/replay/run.sh 41007    # passes with the official fix
python3 agent/run.py --bugs 41007 --condition A               # agent attempt, trace in runs/<timestamp>-A/
```
Last verified end to end on 2026-10-02 (`runs/e2e_readme.out`).